# Late recurrence (>5 years) as anomaly detection

Goal: see whether unsupervised/anomaly-detection methods can prioritize patients who
only recur after 5 years. Loads a patient-level dataset, builds a binary late-recurrence
label (event occurred, time-to-event > 5y), trains unsupervised models on the non-late
patients, then scores everyone and checks how well the late recurrences get captured.

In [5]:
import warnings
warnings.filterwarnings("ignore")

import sys
import os
sys.path.append(os.path.abspath('../../'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score
from functions.repeated_cv import *

RANDOM_STATE = 12
np.random.seed(RANDOM_STATE)

In [6]:
dataset = pd.read_csv("../data/data_cox/data.csv")

diagnosis_variables = []

with open("../data/data_cox/diagnosis.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../data/data_cox/treatment.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

# continuous_features = feature_columns
continuous_features =  ['Age', 'Ki67 positive cells (%) at CB', 'Conventional RT fraction', 'Total administered dose', 'Boost dose administered']
continuous_features = [col for col in continuous_features if col in feature_columns]

In [3]:
from sklearn.preprocessing import MinMaxScaler
from sksurv.util import Surv

# Cap survival days at 1825 and set event status to 0 for censored cases
survival_dataset = survival_dataset.loc[survival_dataset['Survival days'] >= 1826].copy()
survival_dataset['Survival days'] = survival_dataset['Survival days'] - 1826

# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)
groups = survival_dataset['PERSON_ID']

x_train = data_x
y_train = data_y

sum(y_train['event'])

np.int64(6)

In [7]:
from scipy.stats import mannwhitneyu, fisher_exact
from sklearn.preprocessing import StandardScaler

# Case-Control Characterization
RANDOM_STATE_CASE_CONTROL = 12
rng_case_control = np.random.default_rng(RANDOM_STATE_CASE_CONTROL)

# Start from full (untruncated) survival data.
base_df = dataset.copy()
if 'Unnamed: 0' in base_df.columns:
    base_df = base_df.drop(columns=['Unnamed: 0'])

event_col = 'Regional LN recurrence'
time_col = 'Survival days'

# Robust column discovery for key features.
age_candidates = [c for c in base_df.columns if c.strip().lower() == 'age']
ki67_candidates = [c for c in base_df.columns if 'ki67' in c.lower()]
grade_candidates = [c for c in base_df.columns if 'grade' in c.lower()]

if len(age_candidates) == 0:
    raise ValueError('Could not find an Age column.')
if len(ki67_candidates) == 0:
    raise ValueError('Could not find a Ki67 column.')
if len(grade_candidates) == 0:
    raise ValueError('Could not find a Grade column.')

age_col = age_candidates[0]
ki67_col = ki67_candidates[0]
grade_col = grade_candidates[0]

analysis_cols = [event_col, time_col, age_col, ki67_col, grade_col]
if 'PERSON_ID' in base_df.columns:
    analysis_cols = ['PERSON_ID'] + analysis_cols

df_cc = base_df[analysis_cols].copy()
df_cc[event_col] = pd.to_numeric(df_cc[event_col], errors='coerce').fillna(0).astype(int)
df_cc[time_col] = pd.to_numeric(df_cc[time_col], errors='coerce')
df_cc[age_col] = pd.to_numeric(df_cc[age_col], errors='coerce')
df_cc[ki67_col] = pd.to_numeric(df_cc[ki67_col], errors='coerce')

# Keep rows with required fields for matching/tests.
df_cc = df_cc.dropna(subset=[time_col, age_col, ki67_col, grade_col]).copy()
df_cc[grade_col] = df_cc[grade_col].astype(str).str.strip()

# Define groups requested by user.
late_cases = df_cc[(df_cc[event_col] == 1) & (df_cc[time_col] > 5 * 365)].copy()
super_survivors_pool = df_cc[(df_cc[event_col] == 0) & (df_cc[time_col] >= 10 * 365)].copy()

if len(late_cases) < 6:
    raise ValueError(f'Expected at least 6 late-recurrence patients, found {len(late_cases)}.')
if len(super_survivors_pool) < 12:
    raise ValueError(f'Expected at least 12 super-survivors, found {len(super_survivors_pool)}.')

# Use exactly 6 cases, sampled reproducibly if there are more.
if len(late_cases) > 6:
    case_idx = rng_case_control.choice(late_cases.index.to_numpy(), size=6, replace=False)
    late_cases = late_cases.loc[case_idx].copy()

late_cases = late_cases.reset_index(drop=True)

def match_controls_greedy(cases_df, controls_df, ratio=2):
    """Greedy nearest-neighbor matching (without replacement), preferring same Grade."""
    required_controls = ratio * len(cases_df)
    if len(controls_df) < required_controls:
        raise ValueError(
            f'Not enough controls for 1:{ratio} matching. Need {required_controls}, found {len(controls_df)}.'
        )

    controls_work = controls_df.copy()
    num_cols = [age_col, ki67_col]

    # Scale numeric dimensions on control pool for distance computation.
    scaler = StandardScaler()
    controls_scaled = scaler.fit_transform(controls_work[num_cols])
    controls_scaled_df = pd.DataFrame(
        controls_scaled, index=controls_work.index, columns=[f'{c}_z' for c in num_cols]
    )
    controls_work = controls_work.join(controls_scaled_df)

    cases_scaled = scaler.transform(cases_df[num_cols])
    cases_work = cases_df.copy()
    cases_work[[f'{c}_z' for c in num_cols]] = cases_scaled

    selected_control_idx = []

    for case_i in range(len(cases_work)):
        case_row = cases_work.iloc[case_i]
        for _ in range(ratio):
            remaining = controls_work.loc[~controls_work.index.isin(selected_control_idx)].copy()
            if remaining.empty:
                raise ValueError('Ran out of controls during matching.')

            same_grade = remaining[remaining[grade_col] == case_row[grade_col]]
            candidate_pool = same_grade if len(same_grade) > 0 else remaining

            d = np.sqrt(
                (candidate_pool[f'{age_col}_z'] - case_row[f'{age_col}_z']) ** 2
                + (candidate_pool[f'{ki67_col}_z'] - case_row[f'{ki67_col}_z']) ** 2
            )
            best_idx = d.idxmin()
            selected_control_idx.append(best_idx)

    matched_controls = controls_work.loc[selected_control_idx, cases_df.columns].copy()
    matched_controls = matched_controls.reset_index(drop=True)
    return matched_controls

def summarize_and_test(cases_df, controls_df, label='1:2'):
    rows = []

    # Wilcoxon rank-sum analogue for independent samples (Mann-Whitney U).
    for var in [age_col, ki67_col]:
        x = pd.to_numeric(cases_df[var], errors='coerce').dropna().values
        y = pd.to_numeric(controls_df[var], errors='coerce').dropna().values
        stat, p = mannwhitneyu(x, y, alternative='two-sided')
        rows.append({
            'matching': label,
            'feature': var,
            'test': 'Mann-Whitney U (Wilcoxon rank-sum)',
            'cases_median': float(np.median(x)),
            'controls_median': float(np.median(y)),
            'statistic': float(stat),
            'p_value': float(p),
        })

    # Fisher's exact on a binary Grade split: High (contains '3' or 'high') vs Other.
    case_grade_high = cases_df[grade_col].str.lower().str.contains(r'3|high', regex=True).astype(int)
    ctrl_grade_high = controls_df[grade_col].str.lower().str.contains(r'3|high', regex=True).astype(int)

    a = int((case_grade_high == 1).sum())
    b = int((case_grade_high == 0).sum())
    c = int((ctrl_grade_high == 1).sum())
    d = int((ctrl_grade_high == 0).sum())

    table = np.array([[a, b], [c, d]])
    _, p_fisher = fisher_exact(table, alternative='two-sided')

    rows.append({
        'matching': label,
        'feature': f'{grade_col} (High vs Other)',
        'test': "Fisher's exact",
        'cases_median': np.nan,
        'controls_median': np.nan,
        'statistic': np.nan,
        'p_value': float(p_fisher),
    })

    return pd.DataFrame(rows), table

# Perform both requested matching sizes: 12 controls (1:2) and 18 controls (1:3).
matched_ctrl_12 = match_controls_greedy(late_cases, super_survivors_pool, ratio=2)
matched_ctrl_18 = match_controls_greedy(late_cases, super_survivors_pool, ratio=3)

results_12, fisher_table_12 = summarize_and_test(late_cases, matched_ctrl_12, label='1:2 (6 vs 12)')
results_18, fisher_table_18 = summarize_and_test(late_cases, matched_ctrl_18, label='1:3 (6 vs 18)')

case_control_results = pd.concat([results_12, results_18], ignore_index=True)

print('Cohort sizes:')
print(f'Late recurrence cases: {len(late_cases)}')
print(f'Super-survivors pool: {len(super_survivors_pool)}')
print('')

print('Fisher contingency table (1:2) [[Cases High, Cases Other], [Controls High, Controls Other]]:')
print(fisher_table_12)
print('')
print('Fisher contingency table (1:3) [[Cases High, Cases Other], [Controls High, Controls Other]]:')
print(fisher_table_18)
print('')

display(case_control_results)

Cohort sizes:
Late recurrence cases: 6
Super-survivors pool: 44

Fisher contingency table (1:2) [[Cases High, Cases Other], [Controls High, Controls Other]]:
[[ 0  6]
 [ 0 12]]

Fisher contingency table (1:3) [[Cases High, Cases Other], [Controls High, Controls Other]]:
[[ 0  6]
 [ 0 18]]



,matching,feature,test,cases_median,controls_median,statistic,p_value
0,1:2 (6 vs 12),Age,Mann-Whitney U (Wilcoxon rank-sum),49.5,54.0,30.5,0.638882
1,1:2 (6 vs 12),Ki67 positive cells (%) at CB,Mann-Whitney U (Wilcoxon rank-sum),25.0,25.0,36.5,1.000000
2,1:2 (6 vs 12),Grade at CB_2.0 (High vs Other),Fisher's exact,NaN,NaN,NaN,1.000000
3,1:3 (6 vs 18),Age,Mann-Whitney U (Wilcoxon rank-sum),49.5,54.0,42.5,0.462771
4,1:3 (6 vs 18),Ki67 positive cells (%) at CB,Mann-Whitney U (Wilcoxon rank-sum),25.0,25.0,56.5,0.873382
5,1:3 (6 vs 18),Grade at CB_2.0 (High vs Other),Fisher's exact,NaN,NaN,NaN,1.000000


In [9]:
# Late cases (n=6) vs Rest of population characterization
from scipy.stats import mannwhitneyu, fisher_exact

# Build variable list from full dataset: core + estrogen-receptor-like + treatment variables.
full_df = base_df.copy()

event_col_local = 'Regional LN recurrence'
time_col_local = 'Survival days'

er_candidates = [
    c for c in full_df.columns
    if ('oestrogen' in c.lower()) or ('estrogen' in c.lower()) or (c.strip().lower() in ['er', 'er status'])
]
treatment_candidates = [c for c in treatment_variables if c in full_df.columns]

candidate_vars = [age_col, ki67_col, grade_col] + er_candidates + treatment_candidates
compare_vars = list(dict.fromkeys([v for v in candidate_vars if v in full_df.columns]))

needed_cols = [event_col_local, time_col_local] + compare_vars
if 'PERSON_ID' in full_df.columns:
    needed_cols = ['PERSON_ID'] + needed_cols

work_df = full_df[needed_cols].copy()
work_df[event_col_local] = pd.to_numeric(work_df[event_col_local], errors='coerce').fillna(0).astype(int)
work_df[time_col_local] = pd.to_numeric(work_df[time_col_local], errors='coerce')

# Use the same 6 late cases selected in Cell 5 when possible.
if 'PERSON_ID' in work_df.columns and 'PERSON_ID' in late_cases.columns:
    case_ids = set(late_cases['PERSON_ID'].dropna().tolist())
    cases_vs_rest_cases = work_df[work_df['PERSON_ID'].isin(case_ids)].copy()
    cases_vs_rest_rest = work_df[~work_df['PERSON_ID'].isin(case_ids)].copy()
else:
    # Fallback by survival definition + reproducible sampling.
    late_mask = (work_df[event_col_local] == 1) & (work_df[time_col_local] > 5 * 365)
    late_pool = work_df[late_mask].copy()
    if len(late_pool) < 6:
        raise ValueError(f'Expected at least 6 late-recurrence patients, found {len(late_pool)}.')
    cases_vs_rest_cases = late_pool.sample(n=6, random_state=RANDOM_STATE_CASE_CONTROL).copy()
    cases_vs_rest_rest = work_df.loc[~work_df.index.isin(cases_vs_rest_cases.index)].copy()

# Keep one row per case if PERSON_ID duplicates appear.
if 'PERSON_ID' in cases_vs_rest_cases.columns:
    cases_vs_rest_cases = cases_vs_rest_cases.drop_duplicates(subset=['PERSON_ID']).copy()
if len(cases_vs_rest_cases) > 6:
    cases_vs_rest_cases = cases_vs_rest_cases.sample(n=6, random_state=RANDOM_STATE_CASE_CONTROL).copy()

print('Case vs Rest cohort sizes:')
print(f'Late-recurrence cases: {len(cases_vs_rest_cases)}')
print(f'Rest of population: {len(cases_vs_rest_rest)}')
print('')

rows_case_vs_rest = []
skipped_vars = []

for var in compare_vars:
    case_series = cases_vs_rest_cases[var]
    rest_series = cases_vs_rest_rest[var]

    # Numeric variables with >2 distinct values -> Mann-Whitney U.
    case_num = pd.to_numeric(case_series, errors='coerce').dropna()
    rest_num = pd.to_numeric(rest_series, errors='coerce').dropna()

    if (len(case_num) >= 2 and len(rest_num) >= 2 and (case_num.nunique(dropna=True) > 2 or rest_num.nunique(dropna=True) > 2)):
        stat, p = mannwhitneyu(case_num.values, rest_num.values, alternative='two-sided')
        rows_case_vs_rest.append({
            'comparison': '6 late cases vs rest',
            'feature': var,
            'test': 'Mann-Whitney U (Wilcoxon rank-sum)',
            'n_cases': int(len(case_num)),
            'n_rest': int(len(rest_num)),
            'cases_median': float(np.median(case_num.values)),
            'rest_median': float(np.median(rest_num.values)),
            'statistic': float(stat),
            'p_value': float(p),
        })
        continue

    # Binary categorical variables -> Fisher's exact.
    case_cat = case_series.dropna().astype(str).str.strip()
    rest_cat = rest_series.dropna().astype(str).str.strip()
    all_levels = list(pd.unique(pd.concat([case_cat, rest_cat], axis=0)))

    if len(all_levels) == 2 and len(case_cat) > 0 and len(rest_cat) > 0:
        lv1, lv2 = all_levels[0], all_levels[1]
        a = int((case_cat == lv1).sum())
        b = int((case_cat == lv2).sum())
        c = int((rest_cat == lv1).sum())
        d = int((rest_cat == lv2).sum())
        table = np.array([[a, b], [c, d]])
        _, p = fisher_exact(table, alternative='two-sided')
        rows_case_vs_rest.append({
            'comparison': '6 late cases vs rest',
            'feature': f"{var} ({lv1} vs {lv2})",
            'test': "Fisher's exact",
            'n_cases': int(len(case_cat)),
            'n_rest': int(len(rest_cat)),
            'cases_median': np.nan,
            'rest_median': np.nan,
            'statistic': np.nan,
            'p_value': float(p),
        })
    else:
        skipped_vars.append(var)

case_vs_rest_results = (
    pd.DataFrame(rows_case_vs_rest)
    .sort_values('p_value', na_position='last')
    .reset_index(drop=True)
 )

display(case_vs_rest_results)

if len(skipped_vars) > 0:
    print('Variables skipped (not binary categorical and not continuous-ready):')
    print(', '.join(skipped_vars))

Case vs Rest cohort sizes:
Late-recurrence cases: 6
Rest of population: 3221



,comparison,feature,test,n_cases,n_rest,cases_median,rest_median,statistic,p_value
0,6 late cases vs rest,Radiotherapy on chest wall (0.0 vs 1.0),Fisher's exact,6,3221,NaN,NaN,NaN,0.003151
1,6 late cases vs rest,Boost dose administered,Mann-Whitney U (Wilcoxon rank-sum),6,3221,0.0,3.0,4385.0,0.004994
2,6 late cases vs rest,Endocrine therapy performed (0 vs 1),Fisher's exact,6,3221,NaN,NaN,NaN,0.009963
3,6 late cases vs rest,Total administered dose,Mann-Whitney U (Wilcoxon rank-sum),6,3221,0.0,41.0,4969.0,0.010607
4,6 late cases vs rest,Conventional RT fraction,Mann-Whitney U (Wilcoxon rank-sum),6,3221,0.0,3.0,5187.0,0.013993
5,6 late cases vs rest,Treatment in association with chemotherapy (1 ...,Fisher's exact,6,3221,NaN,NaN,NaN,0.033421
6,6 late cases vs rest,Age,Mann-Whitney U (Wilcoxon rank-sum),6,3221,49.5,54.0,7791.0,0.411617
7,6 late cases vs rest,Ki67 positive cells (%) at CB,Mann-Whitney U (Wilcoxon rank-sum),6,3221,25.0,25.0,7949.5,0.447943
8,6 late cases vs rest,Adjuvant therapy using biological drugs (0 vs 1),Fisher's exact,6,3221,NaN,NaN,NaN,0.528739
9,6 late cases vs rest,Adjuvant chemotherapy (0 vs 1),Fisher's exact,6,3221,NaN,NaN,NaN,0.574485


In [10]:
# Quick views: most different variables + estrogen/treatment subsets
display(case_vs_rest_results.head(20))

feature_lower = case_vs_rest_results['feature'].astype(str).str.lower()
er_view = case_vs_rest_results[feature_lower.str.contains('oestrogen|estrogen|\ber\b', regex=True)]
treatment_view = case_vs_rest_results[case_vs_rest_results['feature'].astype(str).isin(treatment_candidates)]

print('Estrogen-receptor related rows:')
display(er_view if len(er_view) > 0 else pd.DataFrame({'info': ['No estrogen-related variable found/testable with current rules.']}))

print('Treatment-variable rows:')
display(treatment_view if len(treatment_view) > 0 else pd.DataFrame({'info': ['No treatment variable was testable with current rules.']}))

,comparison,feature,test,n_cases,n_rest,cases_median,rest_median,statistic,p_value
0,6 late cases vs rest,Radiotherapy on chest wall (0.0 vs 1.0),Fisher's exact,6,3221,NaN,NaN,NaN,0.003151
1,6 late cases vs rest,Boost dose administered,Mann-Whitney U (Wilcoxon rank-sum),6,3221,0.0,3.0,4385.0,0.004994
2,6 late cases vs rest,Endocrine therapy performed (0 vs 1),Fisher's exact,6,3221,NaN,NaN,NaN,0.009963
3,6 late cases vs rest,Total administered dose,Mann-Whitney U (Wilcoxon rank-sum),6,3221,0.0,41.0,4969.0,0.010607
4,6 late cases vs rest,Conventional RT fraction,Mann-Whitney U (Wilcoxon rank-sum),6,3221,0.0,3.0,5187.0,0.013993
5,6 late cases vs rest,Treatment in association with chemotherapy (1 ...,Fisher's exact,6,3221,NaN,NaN,NaN,0.033421
6,6 late cases vs rest,Age,Mann-Whitney U (Wilcoxon rank-sum),6,3221,49.5,54.0,7791.0,0.411617
7,6 late cases vs rest,Ki67 positive cells (%) at CB,Mann-Whitney U (Wilcoxon rank-sum),6,3221,25.0,25.0,7949.5,0.447943
8,6 late cases vs rest,Adjuvant therapy using biological drugs (0 vs 1),Fisher's exact,6,3221,NaN,NaN,NaN,0.528739
9,6 late cases vs rest,Adjuvant chemotherapy (0 vs 1),Fisher's exact,6,3221,NaN,NaN,NaN,0.574485


Estrogen-receptor related rows:


,comparison,feature,test,n_cases,n_rest,cases_median,rest_median,statistic,p_value
10,6 late cases vs rest,Oestrogen receptor status at CB_1.0 (1.0 vs 0.0),Fisher's exact,6,3221,NaN,NaN,NaN,0.599129
15,6 late cases vs rest,Oestrogen receptor status at CB_0.0 (0.0 vs 1.0),Fisher's exact,6,3221,NaN,NaN,NaN,1.000000


Treatment-variable rows:


,comparison,feature,test,n_cases,n_rest,cases_median,rest_median,statistic,p_value
1,6 late cases vs rest,Boost dose administered,Mann-Whitney U (Wilcoxon rank-sum),6,3221,0.0,3.0,4385.0,0.004994
3,6 late cases vs rest,Total administered dose,Mann-Whitney U (Wilcoxon rank-sum),6,3221,0.0,41.0,4969.0,0.010607
4,6 late cases vs rest,Conventional RT fraction,Mann-Whitney U (Wilcoxon rank-sum),6,3221,0.0,3.0,5187.0,0.013993


In [32]:
from sklearn.base import clone
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import MinMaxScaler
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

# Use existing train data from previous cells.
X_source = x_train.copy() if "x_train" in globals() else data_x.copy()
y_source = y_train if "y_train" in globals() else data_y

# Encode categoricals once to keep a stable design matrix across folds.
X_loo = pd.get_dummies(X_source, drop_first=False).astype(float)

# Reuse your Cox hyperparameters when available.
cox_params = globals().get(
    "best_params_for_calibration",
    {
        "alphas": [0.000464159],
        "l1_ratio": 1.0,
        "fit_baseline_model": True,
    },
)

# Columns that should be scaled fold-by-fold (no leakage).
if "continuous_features" in globals() and len(continuous_features) > 0:
    scale_cols = [c for c in continuous_features if c in X_loo.columns]
else:
    scale_cols = X_loo.columns.tolist()

loo = LeaveOneOut()
oof_risk = np.full(X_loo.shape[0], np.nan, dtype=float)
failed_folds = []

for fold_idx, (tr_idx, te_idx) in enumerate(loo.split(X_loo), start=1):
    X_tr = X_loo.iloc[tr_idx].copy()
    X_te = X_loo.iloc[te_idx].copy()
    y_tr = y_source[tr_idx]

    # Fit scaler only on training fold.
    if len(scale_cols) > 0:
        scaler_fold = MinMaxScaler()
        X_tr.loc[:, scale_cols] = scaler_fold.fit_transform(X_tr[scale_cols])
        X_te.loc[:, scale_cols] = scaler_fold.transform(X_te[scale_cols])

    model_fold = CoxnetSurvivalAnalysis(**cox_params)

    try:
        model_fold.fit(X_tr, y_tr)
        oof_risk[te_idx[0]] = float(model_fold.predict(X_te)[0])
    except Exception as exc:
        failed_folds.append((fold_idx, str(exc)))

valid_mask = np.isfinite(oof_risk)
n_valid = int(valid_mask.sum())

if n_valid < 2:
    raise ValueError("Too few valid LOOCV predictions to compute c-index.")

y_valid = y_source[valid_mask]
risk_valid = oof_risk[valid_mask]

c_index_loo = float(concordance_index_censored(
    y_valid["event"],
    y_valid["time"],
    risk_valid,
 )[0])

# Bootstrap distribution of c-index from OOF predictions.
N_BOOTSTRAP_STD = 1000
rng = np.random.default_rng(42)
c_index_boot = []

for b in range(N_BOOTSTRAP_STD):
    bs_idx = rng.integers(0, n_valid, size=n_valid)
    y_bs = y_valid[bs_idx]
    risk_bs = risk_valid[bs_idx]
    try:
        c_bs = concordance_index_censored(
            y_bs["event"],
            y_bs["time"],
            risk_bs,
        )[0]
        c_index_boot.append(float(c_bs))
    except Exception:
        continue

# Save all bootstrap run results for inspection/export.
loo_cindex_bootstrap_df = pd.DataFrame({
    "bootstrap_iter": np.arange(1, len(c_index_boot) + 1),
    "c_index": c_index_boot,
})

c_index_std = float(np.nanstd(c_index_boot, ddof=1)) if len(c_index_boot) > 1 else np.nan
c_index_ci_low = float(np.nanpercentile(c_index_boot, 2.5)) if len(c_index_boot) > 0 else np.nan
c_index_ci_high = float(np.nanpercentile(c_index_boot, 97.5)) if len(c_index_boot) > 0 else np.nan

loo_cox_results_df = pd.DataFrame([
    {
        "n_samples": int(X_loo.shape[0]),
        "n_valid_predictions": n_valid,
        "failed_folds": int(len(failed_folds)),
        "c_index_loocv": c_index_loo,
        "c_index_std_bootstrap": c_index_std,
        "c_index_ci95_low": c_index_ci_low,
        "c_index_ci95_high": c_index_ci_high,
        "n_bootstrap_std": int(len(c_index_boot)),
    }
])

# Optional long-format CI summary table.
loo_cox_ci_df = pd.DataFrame([
    {"metric": "c_index", "estimate": c_index_loo, "std_bootstrap": c_index_std, "ci95_low": c_index_ci_low, "ci95_high": c_index_ci_high}
])

display(loo_cox_results_df)
display(loo_cox_ci_df)
display(loo_cindex_bootstrap_df.head(10))

print(f"LOOCV Cox c-index: {c_index_loo:.4f}")
print(f"Std (bootstrap): {c_index_std:.4f}")
print(f"95% CI: [{c_index_ci_low:.4f}, {c_index_ci_high:.4f}]")

if len(failed_folds) > 0:
    print("First fold errors:")
    for fold_id, msg in failed_folds[:5]:
        print(f"  Fold {fold_id}: {msg}")

# Uncomment to persist all bootstrap runs to disk.
# loo_cindex_bootstrap_df.to_csv("loo_cindex_bootstrap_runs.csv", index=False)

,n_samples,n_valid_predictions,failed_folds,c_index_loocv,c_index_std_bootstrap,c_index_ci95_low,c_index_ci95_high,n_bootstrap_std
0,1091,1091,0,0.680715,0.100456,0.565439,1.0,989


,metric,estimate,std_bootstrap,ci95_low,ci95_high
0,c_index,0.680715,0.100456,0.565439,1.0


,bootstrap_iter,c_index
0,1,0.884146
1,2,0.662763
2,3,0.717172
3,4,0.688447
4,5,1.000000
5,6,0.735294
6,7,0.785786
7,8,0.715921
8,9,0.684708
9,10,0.661692


LOOCV Cox c-index: 0.6807
Std (bootstrap): 0.1005
95% CI: [0.5654, 1.0000]


In [9]:
from sklearn.preprocessing import MinMaxScaler
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

# Define time points for calibration analysis (1, 2, 3, 4, 5 years)
time_points = [365, 730, 1095, 1460, 1825]  # in days

# Use the best model parameters found earlier
best_params_for_calibration = {
    'alphas': [0.000464159],
    'l1_ratio': 1.0,
    'fit_baseline_model': True
    }

n_controls=["Full"]


cox_model = CoxnetSurvivalAnalysis(**best_params_for_calibration)
cox_model.fit(x_train_scaled, y_train)

# Compute Harrell's c-index on train and test sets
train_risk = cox_model.predict(x_train_scaled)
c_index_train = concordance_index_censored(
    y_train["event"], y_train["time"], train_risk
)[0]

print(f"Train c-index: {c_index_train:.4f}")

# Build coefficient table
coef_array = cox_model.coef_
if coef_array.ndim == 1:
    coef_values = coef_array
else:
    # best_params_for_calibration has one alpha, so take first column
    coef_values = coef_array[:, 0]

coef_df = pd.DataFrame({
    "feature": x_train_scaled.columns,
    "coefficient": coef_values
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

# Optional: non-zero coefficients only
nonzero_coef_df = coef_df[coef_df["coefficient"] != 0].reset_index(drop=True)
size = len(nonzero_coef_df)

print("All coefficients (sorted by absolute value):")
print(coef_df.to_string(index=False))

print(f"\nNon-zero coefficients ({size}):")
print(nonzero_coef_df.to_string(index=False))

Train c-index: 0.9670
All coefficients (sorted by absolute value):
                                                  feature  coefficient  abs_coefficient
                   Progesterone receptor status at CB_1.0     2.729815         2.729815
Her2 overexpression (with immunohystochemistry) at CB_3.0     1.872254         1.872254
                     Radiotherapy on supraclavicular area    -1.476608         1.476608
                                    Adjuvant chemotherapy    -1.258923         1.258923
                               Radiotherapy on chest wall    -0.573265         0.573265
                              Radiotherapy (RT) performed     0.267696         0.267696
                              Side location of the lesion    -0.049692         0.049692
                                          Grade at CB_3.0     0.010832         0.010832
                                        Isotype at CB_3.0     0.000000         0.000000
                                        Isotype at CB